## Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
  * A Schema, including the name of the tool, a description, and/or argument definitions (often a JSON Schema)
  * A function or co-routine to execute

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model

model = init_chat_model("groq:qwen/qwen3.6-27b",
                        reasoning_format="hidden")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='Parrots don’t actually "talk" in the human sense. What we perceive as talking is **highly advanced vocal mimicry**, driven by a mix of biology, social behavior, and learning. Here’s why they do it:\n\n🔹 **They\'re Vocal Learners**: Like humans, dolphins, songbirds, and some bats, parrots are born with a neurological capacity to listen, imitate, and reproduce sounds. This ability peaks during a critical developmental window when they\'re young.\n\n🔹 **Social Bonding**: In the wild, parrots live in tight-knit flocks where vocalizations help maintain group cohesion, coordinate movement, and strengthen relationships. In captivity, humans become their "flock," and mimicking our speech is their way of bonding, seeking connection, and staying socially integrated.\n\n🔹 **Physical & Neurological Adaptations**: Parrots have a specialized vocal organ called the **syrinx** (located where the trachea splits into the lungs) and exceptional muscle control, allowing precise sound pr

In [4]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in  {location}"

model_with_tools = model.bind_tools([get_weather])

In [6]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'tool_calls': [{'id': '7nmqtzmdw', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 130, 'prompt_tokens': 276, 'total_tokens': 406, 'completion_time': 0.247284198, 'completion_tokens_details': {'reasoning_tokens': 102}, 'prompt_time': 0.020129271, 'prompt_tokens_details': None, 'queue_time': 0.045250193, 'total_time': 0.267413469}, 'model_name': 'qwen/qwen3.6-27b', 'system_fingerprint': 'fp_2f860a3fc2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019ff733-b94a-7ab2-824d-30a5a51e898d-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '7nmqtzmdw', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 276, 'output_tokens': 130, 'total_tokens': 406, 'output_token_details': {'reasoning': 102}}
Tool: get_weather
Args: {'location': 'Boston'}


## Tool Exection Loops

In [ ]:
# Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)



It's currently sunny in Boston.
